# Sample filter: first action after r/TheseFuckingAccounts, accused before 30 Nov 2022

Filters the 100 suspended / 100 active sample from `sample_200_stratified_100_suspended_100_active.ipynb`.

1. **First action after the subreddit existed.** An account's first action is the earliest of Arctic Shift's
   `earliest_post_at`, `earliest_comment_at` and the oldest `created_utc` in its stored history. Accounts are kept when
   this is after 2014-12-21 00:03:16 UTC, when r/TheseFuckingAccounts was created (PRAW `created_utc`).
2. **Accused before 30 November 2022.** Accounts are kept when `first_mention_utc` (first TFA accusation) is before
   2022-11-30 00:00:00 UTC.

Inputs are read-only. Outputs go to `sample_200_filtered_output/` under new names, and `save_new()` refuses to
overwrite an existing file.

In [ ]:
# Libraries, folders and the two cut-off times.
# TFA_CREATED_UTC = when r/TheseFuckingAccounts was created; ACCUSED_BEFORE_UTC = 30 Nov 2022 (ChatGPT release).
import pandas as pd
import numpy as np
import os
import re
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

IN_DIR = 'sample_200_output'
OUT_DIR = 'sample_200_filtered_output'
TFA_CREATED_UTC = 1419120196                                         # 2014-12-21 00:03:16 UTC, PRAW subreddit.created_utc
ACCUSED_BEFORE_UTC = int(pd.Timestamp('2022-11-30', tz='UTC').timestamp())   # 1669766400
os.makedirs(OUT_DIR, exist_ok=True)

# Write a parquet file only if it does not exist yet (never overwrite a saved sample).
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

# The 200-account sample and its stored history.
accounts = pd.read_parquet(os.path.join(IN_DIR, 'sample_accounts.parquet'))
posts = pd.read_parquet(os.path.join(IN_DIR, 'sample_posts.parquet'))
comments = pd.read_parquet(os.path.join(IN_DIR, 'sample_comments.parquet'))
print(accounts['account_status'].value_counts().to_string())

## 1. First action on Reddit

In [ ]:
# First action = earliest of: archive meta first post, archive meta first comment, earliest stored row.
history_first = (pd.concat([posts[['outer_key', 'created_utc']], comments[['outer_key', 'created_utc']]])
                  .groupby('outer_key')['created_utc'].min())
accounts['history_first_utc'] = accounts['username_key'].map(history_first)
accounts['first_action_utc'] = accounts[['arctic_earliest_post_at', 'arctic_earliest_comment_at',
                                         'history_first_utc']].min(axis=1)
assert accounts['first_action_utc'].notna().all()

# the archive _meta and the stored rows should agree; report any account where they do not
gap = (accounts['history_first_utc'] - accounts['first_action_utc'])
print('accounts whose stored history starts later than the archive meta says:', (gap > 0).sum())

# Filter 1: the account's first action came after the subreddit was created.
accounts['first_action_after_tfa'] = accounts['first_action_utc'] > TFA_CREATED_UTC
pd.crosstab(accounts['account_status'], accounts['first_action_after_tfa'], margins=True)

## 2. First accused before 30 November 2022

In [ ]:
# Filter 2: the account was first accused before 30 Nov 2022.
accounts['accused_before_cutoff'] = accounts['first_mention_utc'] < ACCUSED_BEFORE_UTC
pd.crosstab(accounts['account_status'], accounts['accused_before_cutoff'], margins=True)

## 3. Apply both filters

In [ ]:
# How many accounts survive each filter in turn, by status.
steps = pd.DataFrame({
    'sample': accounts['account_status'].value_counts(),
    'first action after TFA created': accounts.loc[accounts['first_action_after_tfa'], 'account_status'].value_counts(),
    '+ accused before 2022-11-30': accounts.loc[accounts['first_action_after_tfa'] & accounts['accused_before_cutoff'],
                                                'account_status'].value_counts(),
}).fillna(0).astype(int)
steps.loc['total'] = steps.sum()
steps

In [ ]:
# Keep accounts that pass both filters, and only their posts and comments.
filtered = accounts[accounts['first_action_after_tfa'] & accounts['accused_before_cutoff']].reset_index(drop=True)
keep = set(filtered['username_key'])
filtered_posts = posts[posts['outer_key'].isin(keep)].reset_index(drop=True)
filtered_comments = comments[comments['outer_key'].isin(keep)].reset_index(drop=True)
print(f'accounts {len(filtered)}   posts {len(filtered_posts):,}   comments {len(filtered_comments):,}')
# Per status: accounts kept, earliest first action, and range of first-accusation dates.
filtered.groupby('account_status').agg(
    accounts=('username_key', 'size'),
    first_action_min=('first_action_utc', lambda s: pd.to_datetime(s.min(), unit='s')),
    first_accused_min=('first_mention_utc', lambda s: pd.to_datetime(s.min(), unit='s')),
    first_accused_max=('first_mention_utc', lambda s: pd.to_datetime(s.max(), unit='s')))

## 4. Save

In [ ]:
# Save the filtered accounts, posts and comments.
save_new(filtered, 'sample_accounts_filtered.parquet')
save_new(filtered_posts, 'sample_posts_filtered.parquet')
save_new(filtered_comments, 'sample_comments_filtered.parquet')